# KLOD 0.6 — Treinamento por professor-aluno

O Qwen3-1.7B atual será o professor congelado. Ele gera demonstrações. O aluno começa do Qwen3-1.7B-Base e aprende com as respostas aprovadas por QLoRA. Isto é destilação supervisionada, não treinamento do zero.

As 48 perguntas iniciais validam o pipeline; elas não bastam para produzir uma IA de domínio geral.


## 1. Ative uma GPU NVIDIA

No Colab, selecione Ambiente de execução → Alterar tipo de ambiente de execução → GPU. Uma GPU com 16 GiB ou mais é um ponto de partida prático.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Ative uma GPU CUDA antes de continuar."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")


## 2. Clone a branch development e instale as dependências


In [ ]:
%cd /content
!rm -rf KLOD-AI
!git clone --depth 1 -b development https://github.com/japaguimarassi-create/Security-lab.git KLOD-AI
%cd /content/KLOD-AI
!pip install -e ".[training]"


## 3. Verifique os parâmetros e os recursos


In [ ]:
!python scripts/training_preflight.py --stage teacher --config configs/training/distillation.yaml


## 4. Gere exemplos do professor

O resultado fica em data/generated/teacher_candidates.jsonl. Todos começam com approved=false. Os filtros automáticos detectam respostas vazias, curtas, longas, repetidas e que apenas ecoam a pergunta; eles não provam que o conteúdo está correto.


In [ ]:
!python scripts/generate_teacher_data.py --config configs/training/distillation.yaml


## 5. Revise e aprove somente respostas corretas

No Colab, revise os exemplos com o código abaixo. Edite approved_ids e inclua apenas os IDs cujas respostas você conferiu. São necessários ao menos 20 para este experimento inicial.


In [ ]:
import json
from pathlib import Path

candidate_file = Path("data/generated/teacher_candidates.jsonl")
candidates = [json.loads(line) for line in candidate_file.read_text(encoding="utf-8").splitlines() if line.strip()]

for item in candidates:
    print("\n" + "=" * 80)
    print(item["id"], item["domain"], item["language"], "quality:", item["quality_passed"])
    print("PERGUNTA:", item["messages"][-2]["content"])
    print("RESPOSTA:", item["messages"][-1]["content"])

approved_ids = []  # Exemplo: ["pt-001", "pt-002"]; inclua apenas respostas revisadas
approved = 0
for item in candidates:
    item["approved"] = item["id"] in approved_ids and item["quality_passed"] is True
    item["review_status"] = "approved" if item["approved"] else "pending_or_rejected"
    if item["approved"]:
        item["review_notes"] = "Reviewed in Colab"
        approved += 1

candidate_file.write_text(
    "".join(json.dumps(item, ensure_ascii=False) + "\n" for item in candidates),
    encoding="utf-8",
)
print("Approved:", approved)
assert approved >= 20, "Revise e aprove pelo menos 20 exemplos antes do treino."


## 6. Exporte e treine o aluno


In [ ]:
!python scripts/approve_teacher_data.py --minimum 20
!python scripts/training_preflight.py --stage sft --config configs/training/distillation_sft.yaml
!python scripts/train_sft.py --config configs/training/distillation_sft.yaml


## 7. Salve o adaptador no Google Drive


In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/KLOD/checkpoints/klod-teacher-distilled")
destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree("outputs/klod-teacher-distilled", destination)
print("Adapter saved to:", destination)


## Depois do primeiro ciclo

O adaptador gerado é um primeiro experimento, não um modelo comprovadamente superior. Amplie a base para milhares de exemplos revisados, crie um conjunto de avaliação separado e compare respostas do aluno e do professor antes de promover o adaptador.
